<a href="https://colab.research.google.com/github/Mr-KanishakGupta/Ai_Lab-1BF24CS131-/blob/main/Week2_8_puzzle(1BF24CS131).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def get_neighbors(state):
    neighbors = []

    index = state.index(0)
    row = index // 3
    col = index % 3

    moves = [
        (-1, 0), # Up
        (1, 0), # Down
        (0, -1), # Left
        (0, 1) # Right
    ]

    for dr, dc in moves:
        new_row = row + dr
        new_col = col + dc

        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_index = new_row * 3 + new_col

            new_state = list(state)
            new_state[index], new_state[new_index] = \
                new_state[new_index], new_state[index]

            neighbors.append(tuple(new_state))

    return neighbors


def depth_limited_search(state, goal, depth, path):

    if state == goal:
        return path

    if depth == 0:
        return None

    for next_state in get_neighbors(state):

        if next_state not in path:
            result = depth_limited_search(
                next_state,
                goal,
                depth - 1,
                path + [next_state]
            )

            if result:
                return result

    return None


def idf(start, goal):

    depth = 0

    while True:
        result = depth_limited_search(
            start,
            goal,
            depth,
            [start]
        )

        if result:
            return result

        depth += 1


def print_puzzle(state):

    for i in range(0, 9, 3):
        print(state[i:i+3])

    print()


# Initial state
start = (
    2, 8, 3,
    1, 6, 4,
    7, 0, 5
)

# Goal state
goal = (
    1, 2, 3,
    8, 0, 4,
    7, 6, 5
)


# Solve the puzzle
solution = idf(start, goal)


print("Solution Found!")
print("Number of moves:", len(solution) - 1)
print()

for step in solution:
    print_puzzle(step)


import math

# Average branching factor for an 8-puzzle
# (4 corners * 2 moves + 4 edges * 3 moves + 1 center * 4 moves) / 9 positions
branching_factor = 3

# Depth of the solution is the number of moves
solution_depth = 5

print(f"Average branching factor (b): {branching_factor:.2f}")
print(f"Depth of the solution (d): {solution_depth}")

try:
    # Calculate approximate time and space complexity (b^d)
    time_space_complexity = math.pow(branching_factor, solution_depth)
    print(f"Approximate time/space complexity (b^d): {time_space_complexity:.2e}")
except OverflowError:
    print(f"Approximate time/space complexity (b^d): Too large to calculate (b={branching_factor:.2f}, d={solution_depth})")


Solution Found!
Number of moves: 5

(2, 8, 3)
(1, 6, 4)
(7, 0, 5)

(2, 8, 3)
(1, 0, 4)
(7, 6, 5)

(2, 0, 3)
(1, 8, 4)
(7, 6, 5)

(0, 2, 3)
(1, 8, 4)
(7, 6, 5)

(1, 2, 3)
(0, 8, 4)
(7, 6, 5)

(1, 2, 3)
(8, 0, 4)
(7, 6, 5)

Average branching factor (b): 3.00
Depth of the solution (d): 5
Approximate time/space complexity (b^d): 2.43e+02


In [ ]:
import sys
sys.setrecursionlimit(2000)

class EightPuzzleIDDFS:
    def __init__(self, initial_grid, goal_grid=None):
        self.initial_grid = tuple(tuple(row) for row in initial_grid)
        if goal_grid is None:
            self.goal_grid = ((1, 2, 3), (8, 0, 4), (7, 6, 5))
        else:
            self.goal_grid = tuple(tuple(row) for row in goal_grid)

        self.visited = set()
        self.solution_path = []
        self.states_path = []  # Tracks the grid state at each step

    def find_blank(self, grid):
        for r in range(3):
            for c in range(3):
                if grid[r][c] == 0:
                    return r, c

    def get_neighbors(self, grid):
        r, c = self.find_blank(grid)
        neighbors = []
        moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

        for dr, dc, move_name in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_grid = [list(row) for row in grid]
                new_grid[r][c], new_grid[nr][nc] = new_grid[nr][nc], new_grid[r][c]
                neighbors.append((tuple(tuple(row) for row in new_grid), move_name))
        return neighbors

    def solve(self, max_limit=50):
        for depth_limit in range(max_limit + 1):
            self.visited.clear()
            self.solution_path = []
            self.states_path = [self.initial_grid]  # Start tracking with the initial state
            if self._dfs(self.initial_grid, 0, depth_limit):
                return self.solution_path, self.states_path
        return None, None

    def _dfs(self, current_grid, depth, depth_limit):
        # 1. Visited Check FIRST
        if current_grid in self.visited:
            return False

        # 2. Goal Check SECOND
        if current_grid == self.goal_grid:
            return True

        if depth > depth_limit:
            return False

        self.visited.add(current_grid)

        for neighbor_grid, move in self.get_neighbors(current_grid):
            if neighbor_grid not in self.visited:
                self.solution_path.append(move)
                self.states_path.append(neighbor_grid)

                if self._dfs(neighbor_grid, depth + 1, depth_limit):
                    return True

                self.solution_path.pop()  # Backtrack move
                self.states_path.pop()    # Backtrack state

        self.visited.remove(current_grid)
        return False

def print_grid(grid):
    """Helper function to print a clean 3x3 layout."""
    for row in grid:
        print("  " + " ".join(str(x) if x != 0 else "_" for x in row))
    print()

# =====================================================================
# Execution with step-by-step printing
# =====================================================================
start_grid = [
    [2, 8, 3],
    [1, 6, 4],
    [7, 0, 5]
]

solver = EightPuzzleIDDFS(start_grid)
path, states = solver.solve()

if path and states:
    print(f"--- Initial State ---")
    print_grid(states[0])

    for i in range(len(path)):
        print(f"Step {i+1}: Move {path[i]}")
        print_grid(states[i+1])

    print(f"Solution Found in {len(path)} steps!")
else:
    print("\nNo solution found.")


--- Initial State ---
  2 8 3
  1 6 4
  7 _ 5

Step 1: Move Up
  2 8 3
  1 _ 4
  7 6 5

Step 2: Move Up
  2 _ 3
  1 8 4
  7 6 5

Step 3: Move Left
  _ 2 3
  1 8 4
  7 6 5

Step 4: Move Down
  1 2 3
  _ 8 4
  7 6 5

Step 5: Move Right
  1 2 3
  8 _ 4
  7 6 5

Solution Found in 5 steps! (Optimal)
